# Phase 3: Model Training & Comparison

**Purpose**: Train and evaluate 4 ML algorithms on Phase 2 features

**Approach**:
1. Load Phase 2 feature dataset (88,190 records, 30 features)
2. Grouped stratified split (case-based, keep entire datasets together)
3. Train 4 algorithms: Random Forest, XGBoost, LightGBM, Logistic Regression
4. Evaluate using forensic-appropriate metrics (F1, Recall, Precision, FNR, FPR)
5. Compare results and select best algorithm for Phase 4 hyperparameter tuning

**Methodology**: Fair comparison with reasonable defaults for all algorithms

**Key Challenge**: Extreme class imbalance (0.3% suspicious, 99.7% benign)

## Setup and Configuration

**Dataset**: Phase 2 output (all_cases_combined_features.csv)
- 88,190 files from 18 training datasets (12 PE + 6 APT training)
- 266 suspicious files (0.3%)
- 87,924 benign files (99.7%)
- 30 engineered features

**Train/Test Split Strategy**: Grouped Stratified (Case-Based)
- Keep entire datasets together (no file-level splitting within datasets)
- Training: ~14-15 datasets
- Testing: ~3-4 datasets
- Rationale: Tests generalization to NEW attack scenarios (production deployment scenario)

**Algorithms**:
1. Random Forest - Ensemble baseline
2. XGBoost - Gradient boosting
3. LightGBM - Fast gradient boosting
4. Logistic Regression - Linear baseline

**Evaluation Metrics** (appropriate for extreme imbalance):
1. F1-Score - Harmonic mean of precision and recall
2. Recall (Detection Rate) - Of all attacks, how many caught?
3. Precision - Of flagged files, how many are real attacks?
4. False Negative Rate (FNR) - What % of attacks missed? (security risk)
5. False Positive Rate (FPR) - What % of benign files flagged? (analyst workload)

**Note**: Accuracy and AUC-ROC omitted due to extreme imbalance (naive "all benign" model achieves 99.7% accuracy)


In [219]:
# Cell 2: Imports

import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    confusion_matrix, classification_report
)

# Gradient boosting libraries
import xgboost as xgb
import lightgbm as lgb

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully")


All libraries imported successfully


In [220]:
# Cell 3: Load Phase 2 Features

# Define paths
base_dir = Path('/Users/soni/Github/Digital-Detectives_Thesis')
features_path = base_dir / 'data/processed/Phase 2 - Features/all_cases_combined_features.csv'
output_dir = base_dir / 'data/processed/Phase 3 - Model Training'
output_dir.mkdir(parents=True, exist_ok=True)

# Load features
print("Loading Phase 2 feature dataset...")
df = pd.read_csv(features_path)

print(f"\nDataset loaded successfully:")
print(f"  Total files: {len(df):,}")
print(f"  Total columns: {len(df.columns)}")
print(f"  Suspicious files: {df['is_flagged_suspicious'].sum():,}")
print(f"  Benign files: {(~df['is_flagged_suspicious']).sum():,}")
print(f"  Suspicious rate: {(df['is_flagged_suspicious'].sum() / len(df) * 100):.3f}%")

# Display columns
print(f"\nColumns ({len(df.columns)} total):")
print(df.columns.tolist())

# Display first few rows
print("\nFirst 3 rows:")
df.head(3)


Loading Phase 2 feature dataset...

Dataset loaded successfully:
  Total files: 88,190
  Total columns: 52
  Suspicious files: 266
  Benign files: 87,924
  Suspicious rate: 0.302%

Columns (52 total):
['dataset', 'filename', 'full_path', 'lf_event', 'lf_detail', 'lf_event_time', 'usn_event_info', 'usn_timestamp', 'suspicious_detail_lf', 'suspicious_detail_usn', 'has_logfile_suspicious', 'has_usnjrnl_suspicious', 'cross_artifact_detected', 'is_flagged_suspicious', 'ground_truth_label', 'zero_in_nanoseconds_lf', 'zero_in_nanoseconds_suspicious', 'zero_in_nanoseconds', 'time_reversal_event', 'basic_info_changed', 'using_another_timestamp', 'si_timestamp_changed', 'update_resident_value', 'creation_time_modified', 'modified_time_modified', 'accessed_time_modified', 'mft_time_modified', 'timestamp_changed_to_past', 'multiple_timestamps_changed', 'same_as_another_file', 'zero_nano_time_reversal', 'has_logfile_evidence', 'has_usnjrnl_evidence', 'cross_artifact_validation_score', 'is_executabl

,dataset,filename,full_path,lf_event,lf_detail,lf_event_time,usn_event_info,usn_timestamp,suspicious_detail_lf,suspicious_detail_usn,...,in_program_files,has_timestamp_data,timestamp_source,is_windows_appraiser,is_installer_temp_file,is_windows_update_temp,is_user_document,is_user_media,is_in_cloud_sync_folder,user_file_risk_score
0,01-PE,$LogFile,\Users\blueangel\Desktop\Tools\$LogFile,NaN,NaN,NaN,Basic_Info_Changed / File_Closed,12/23/23 00:23:39,NaN,NaN,...,0,1,2,0,0,0,False,False,0,0.0
1,01-PE,$dpx$.tmp,\Windows\SoftwareDistribution\Download\574ea5e...,NaN,NaN,NaN,Basic_Info_Changed / Content_Indexed_Attr_Chan...,12/19/23 15:23:43,NaN,NaN,...,0,1,2,0,0,0,False,False,0,0.0
2,01-PE,00020000000208812731780A,NaN,NaN,NaN,NaN,Basic_Info_Changed / File_Renamed_New / File_C...,12/19/23 15:24:39,NaN,NaN,...,0,1,2,0,0,0,False,False,0,0.0


In [221]:
# Cell 4: Analyze Suspicious Distribution by Dataset

print("Suspicious File Distribution Across 18 Training Datasets:")
print("-" * 80)

# Group by dataset
dataset_stats = df.groupby('dataset').agg({
    'is_flagged_suspicious': ['sum', 'count']
}).reset_index()

dataset_stats.columns = ['dataset', 'suspicious_count', 'total_files']
dataset_stats['benign_count'] = dataset_stats['total_files'] - dataset_stats['suspicious_count']
dataset_stats['suspicious_pct'] = (dataset_stats['suspicious_count'] / dataset_stats['total_files'] * 100).round(2)

# Sort by suspicious count (descending)
dataset_stats = dataset_stats.sort_values('suspicious_count', ascending=False)

print(dataset_stats.to_string(index=False))

print("\n" + "=" * 80)
print(f"Total suspicious files: {dataset_stats['suspicious_count'].sum()}")
print(f"Total benign files: {dataset_stats['benign_count'].sum()}")
print(f"Total files: {dataset_stats['total_files'].sum()}")
print(f"Overall suspicious rate: {(dataset_stats['suspicious_count'].sum() / dataset_stats['total_files'].sum() * 100):.3f}%")

# Datasets with no suspicious files
no_suspicious = dataset_stats[dataset_stats['suspicious_count'] == 0]
if len(no_suspicious) > 0:
    print(f"\nDatasets with NO suspicious files ({len(no_suspicious)}):")
    print(no_suspicious['dataset'].tolist())


Suspicious File Distribution Across 18 Training Datasets:
--------------------------------------------------------------------------------
        dataset  suspicious_count  total_files  benign_count  suspicious_pct
          06-PE                69          896           827            7.70
          12-PE                68          902           834            7.54
          09-PE                36         8896          8860            0.40
          08-PE                32         8798          8766            0.36
          10-PE                31         8893          8862            0.35
       05-APT29                 7         4695          4688            0.15
11-DarkHotelbbd                 3         3495          3492            0.09
       02-APT19                 2         4482          4480            0.04
          02-PE                 2         8682          8680            0.02
          03-PE                 2         8672          8670            0.02
       04-APT2

In [222]:
# Cell 5: Prepare Features and Labels (FIXED - Exclude Label-Encoding Features)

# Define columns to explicitly exclude (metadata, labels, and label-encoding features)
exclude_cols = [
    'dataset', 'filename', 'full_path', 
    # Labels
    'is_flagged_suspicious', 'suspicious_category', 'suspicious_detail',
    'ground_truth_label',  # THIS WAS THE PROBLEM - It's the label itself!
    # Label-encoding features (created from Suspicious CSV)
    'has_usnjrnl_suspicious', 'has_logfile_suspicious',  # These encode the label!
    # Raw data columns
    'lf_lsn', 'lf_event_time', 'lf_event', 'lf_detail', 'lf_creation_time', 
    'lf_modified_time', 'lf_mft_modified_time', 'lf_accessed_time',
    'usn_usn', 'usn_timestamp', 'usn_event_info', 'usn_source_info'
]

# Get initial feature columns
potential_features = [col for col in df.columns if col not in exclude_cols]

# Check data types - identify non-numeric columns
print("Checking feature data types...")
print("-" * 80)

non_numeric_cols = []
for col in potential_features:
    if df[col].dtype == 'object':  # String columns
        non_numeric_cols.append(col)
        print(f"WARNING: Non-numeric column found: '{col}' (dtype: {df[col].dtype})")

if non_numeric_cols:
    print(f"\nExcluding {len(non_numeric_cols)} non-numeric columns:")
    for col in non_numeric_cols:
        print(f"  - {col}")
    feature_cols = [col for col in potential_features if col not in non_numeric_cols]
else:
    feature_cols = potential_features

print(f"\nFinal feature columns ({len(feature_cols)} total):")
print("-" * 80)
for i, col in enumerate(feature_cols, 1):
    print(f"{i:2d}. {col}")

# Prepare X (features) and y (labels)
X = df[feature_cols].copy()
y = df['is_flagged_suspicious'].astype(int).copy()
groups = df['dataset'].copy()

print(f"\nFeature matrix shape: {X.shape}")
print(f"Label distribution:")
print(f"  Benign (0): {(y == 0).sum():,} ({(y == 0).sum() / len(y) * 100:.2f}%)")
print(f"  Suspicious (1): {(y == 1).sum():,} ({(y == 1).sum() / len(y) * 100:.2f}%)")

# Check for missing values
missing = X.isnull().sum().sum()
print(f"\nMissing values in features: {missing}")

if missing > 0:
    print("\nColumns with missing values:")
    print(X.isnull().sum()[X.isnull().sum() > 0])
    print("\nFilling missing values with 0...")
    X = X.fillna(0)

# Final check for label leakage
print("\n" + "=" * 80)
print("Checking for label leakage (features with >90% correlation to label):")
print("-" * 80)
high_corr = X.corrwith(y).abs().sort_values(ascending=False)
suspicious_features = high_corr[high_corr > 0.9]
if len(suspicious_features) > 0:
    print("WARNING: High correlation features detected:")
    print(suspicious_features)
    print("\nThese should probably be excluded!")
else:
    print("No label leakage detected - proceeding with training")


Checking feature data types...
--------------------------------------------------------------------------------

Excluding 2 non-numeric columns:
  - suspicious_detail_lf
  - suspicious_detail_usn

Final feature columns (38 total):
--------------------------------------------------------------------------------
 1. cross_artifact_detected
 2. zero_in_nanoseconds_lf
 3. zero_in_nanoseconds_suspicious
 4. zero_in_nanoseconds
 5. time_reversal_event
 6. basic_info_changed
 7. using_another_timestamp
 8. si_timestamp_changed
 9. update_resident_value
10. creation_time_modified
11. modified_time_modified
12. accessed_time_modified
13. mft_time_modified
14. timestamp_changed_to_past
15. multiple_timestamps_changed
16. same_as_another_file
17. zero_nano_time_reversal
18. has_logfile_evidence
19. has_usnjrnl_evidence
20. cross_artifact_validation_score
21. is_executable
22. is_document
23. is_archive
24. is_image
25. path_depth
26. filename_length
27. in_temp_directory
28. in_system_directory


In [223]:
# Cell 6 REPLACEMENT: Manual Stratified Split

print("Creating manual stratified train/test split (case-based)...")
print("-" * 80)

# Manually select test datasets to ensure balanced suspicious distribution
# Based on Cell 4 analysis, choose datasets with mix of high/low suspicious counts
test_datasets_manual = [
    '06-PE',      # 69 suspicious (high)
    '01-APT17',   # 2 suspicious (low)
    '04-APT28',   # 2 suspicious (low)  
    '05-PE'       # 1 suspicious (low)
]

# All others go to training
train_datasets_manual = df['dataset'].unique()
train_datasets_manual = [d for d in train_datasets_manual if d not in test_datasets_manual]

print(f"Manually selected test datasets: {sorted(test_datasets_manual)}")
print(f"Training datasets: {sorted(train_datasets_manual)}")

# Create train/test splits
train_mask = df['dataset'].isin(train_datasets_manual)
test_mask = df['dataset'].isin(test_datasets_manual)

X_train = X[train_mask].copy()
X_test = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

print(f"\nTraining set:")
print(f"  Total files: {len(X_train):,}")
print(f"  Suspicious: {y_train.sum():,} ({y_train.sum() / len(y_train) * 100:.3f}%)")

print(f"\nTest set:")
print(f"  Total files: {len(X_test):,}")
print(f"  Suspicious: {y_test.sum():,} ({y_test.sum() / len(y_test) * 100:.3f}%)")

print(f"\nTest set suspicious distribution:")
for dataset in sorted(test_datasets_manual):
    dataset_mask = df['dataset'] == dataset
    dataset_suspicious = df[dataset_mask]['is_flagged_suspicious'].sum()
    dataset_total = dataset_mask.sum()
    print(f"  {dataset}: {dataset_suspicious}/{dataset_total}")


Creating manual stratified train/test split (case-based)...
--------------------------------------------------------------------------------
Manually selected test datasets: ['01-APT17', '04-APT28', '05-PE', '06-PE']
Training datasets: ['01-PE', '02-APT19', '02-PE', '03-PE', '04-PE', '05-APT29', '07-PE', '08-PE', '09-PE', '10-DarkHotel663', '10-PE', '11-DarkHotelbbd', '11-PE', '12-PE']

Training set:
  Total files: 77,492
  Suspicious: 192 (0.248%)

Test set:
  Total files: 10,698
  Suspicious: 74 (0.692%)

Test set suspicious distribution:
  01-APT17: 2/4419
  04-APT28: 2/4481
  05-PE: 1/902
  06-PE: 69/896


In [224]:
# Cell 7: Verify Split Quality (CORRECTED - Uses Cell 6 Variables)

# Analyze suspicious distribution in train vs test
train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

train_dataset_stats = train_df.groupby('dataset')['is_flagged_suspicious'].agg(['sum', 'count'])
test_dataset_stats = test_df.groupby('dataset')['is_flagged_suspicious'].agg(['sum', 'count'])

print("Training Datasets - Suspicious Distribution:")
print("-" * 80)
print(train_dataset_stats.to_string())
print(f"\nTotal training suspicious: {train_dataset_stats['sum'].sum()}")

print("\n" + "=" * 80)
print("Test Datasets - Suspicious Distribution:")
print("-" * 80)
print(test_dataset_stats.to_string())
print(f"\nTotal test suspicious: {test_dataset_stats['sum'].sum()}")

# Verify this matches y_test
print(f"\nVerification:")
print(f"  y_train.sum() = {y_train.sum()} (should match training suspicious above)")
print(f"  y_test.sum() = {y_test.sum()} (should match test suspicious above)")

# Check if test set has sufficient suspicious examples
if y_test.sum() < 10:
    print("\nWARNING: Test set has very few suspicious files (<10). Consider adjusting split.")
else:
    print(f"\nSplit quality validated: Test set has {y_test.sum()} suspicious files for evaluation")


Training Datasets - Suspicious Distribution:
--------------------------------------------------------------------------------
                 sum  count
dataset                    
01-PE              2   5931
02-APT19           2   4482
02-PE              2   8682
03-PE              2   8672
04-PE              1    855
05-APT29           7   4695
07-PE              2   8795
08-PE             32   8798
09-PE             36   8896
10-DarkHotel663    2   3501
10-PE             31   8893
11-DarkHotelbbd    3   3495
11-PE              2    895
12-PE             68    902

Total training suspicious: 192

Test Datasets - Suspicious Distribution:
--------------------------------------------------------------------------------
          sum  count
dataset             
01-APT17    2   4419
04-APT28    2   4481
05-PE       1    902
06-PE      69    896

Total test suspicious: 74

Verification:
  y_train.sum() = 192 (should match training suspicious above)
  y_test.sum() = 74 (should match test s

In [225]:
# Cell 8: Calculate Class Weights for Imbalanced Data

# Calculate class weights to handle extreme imbalance
n_benign = (y_train == 0).sum()
n_suspicious = (y_train == 1).sum()
scale_pos_weight = n_benign / n_suspicious

print("Class Imbalance Handling:")
print("-" * 80)
print(f"Benign files (class 0): {n_benign:,}")
print(f"Suspicious files (class 1): {n_suspicious:,}")
print(f"Imbalance ratio: {n_benign / n_suspicious:.1f}:1")
print(f"\nScale positive weight (for XGBoost/LightGBM): {scale_pos_weight:.2f}")
print(f"Class weights (for Random Forest/Logistic Regression): {{0: 1, 1: {scale_pos_weight:.2f}}}")

# Store for model training
class_weights = {0: 1, 1: scale_pos_weight}


Class Imbalance Handling:
--------------------------------------------------------------------------------
Benign files (class 0): 77,300
Suspicious files (class 1): 192
Imbalance ratio: 402.6:1

Scale positive weight (for XGBoost/LightGBM): 402.60
Class weights (for Random Forest/Logistic Regression): {0: 1, 1: 402.60}


In [226]:
# Cell 9: Train Random Forest

print("Training Random Forest...")
print("-" * 80)

# Initialize Random Forest with balanced class weights
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

# Train
rf_model.fit(X_train, y_train)

# Predict
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)
rf_test_proba = rf_model.predict_proba(X_test)[:, 1]

# Show prediction distribution (diagnostic)
print("Random Forest training completed")
print(f"  Training predictions:")
print(f"    Predicted benign: {(rf_train_pred == 0).sum():,}")
print(f"    Predicted suspicious: {(rf_train_pred == 1).sum():,}")
print(f"  Test predictions:")
print(f"    Predicted benign: {(rf_test_pred == 0).sum():,}")
print(f"    Predicted suspicious: {(rf_test_pred == 1).sum():,}")


Training Random Forest...
--------------------------------------------------------------------------------
Random Forest training completed
  Training predictions:
    Predicted benign: 77,057
    Predicted suspicious: 435
  Test predictions:
    Predicted benign: 10,539
    Predicted suspicious: 159


In [227]:
# Cell 10: Train XGBoost

print("Training XGBoost...")
print("-" * 80)

# Initialize XGBoost with scale_pos_weight for imbalance
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

# Train
xgb_model.fit(X_train, y_train, verbose=False)

# Predict
xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)
xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]

# Show prediction distribution (diagnostic)
print("XGBoost training completed")
print(f"  Training predictions:")
print(f"    Predicted benign: {(xgb_train_pred == 0).sum():,}")
print(f"    Predicted suspicious: {(xgb_train_pred == 1).sum():,}")
print(f"  Test predictions:")
print(f"    Predicted benign: {(xgb_test_pred == 0).sum():,}")
print(f"    Predicted suspicious: {(xgb_test_pred == 1).sum():,}")


Training XGBoost...
--------------------------------------------------------------------------------
XGBoost training completed
  Training predictions:
    Predicted benign: 77,057
    Predicted suspicious: 435
  Test predictions:
    Predicted benign: 10,539
    Predicted suspicious: 159


In [228]:
# Cell 11: Train LightGBM

print("Training LightGBM...")
print("-" * 80)

# Initialize LightGBM with balanced class weights
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=-1,
    num_leaves=31,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Train
lgb_model.fit(X_train, y_train)

# Predict
lgb_train_pred = lgb_model.predict(X_train)
lgb_test_pred = lgb_model.predict(X_test)
lgb_test_proba = lgb_model.predict_proba(X_test)[:, 1]

# Show prediction distribution (diagnostic)
print("LightGBM training completed")
print(f"  Training predictions:")
print(f"    Predicted benign: {(lgb_train_pred == 0).sum():,}")
print(f"    Predicted suspicious: {(lgb_train_pred == 1).sum():,}")
print(f"  Test predictions:")
print(f"    Predicted benign: {(lgb_test_pred == 0).sum():,}")
print(f"    Predicted suspicious: {(lgb_test_pred == 1).sum():,}")


Training LightGBM...
--------------------------------------------------------------------------------
LightGBM training completed
  Training predictions:
    Predicted benign: 77,057
    Predicted suspicious: 435
  Test predictions:
    Predicted benign: 10,538
    Predicted suspicious: 160


In [229]:
# Cell 12: Train Logistic Regression

print("Training Logistic Regression...")
print("-" * 80)

# Initialize Logistic Regression with balanced class weights
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    solver='lbfgs',
    n_jobs=-1
)

# Train
lr_model.fit(X_train, y_train)

# Predict
lr_train_pred = lr_model.predict(X_train)
lr_test_pred = lr_model.predict(X_test)
lr_test_proba = lr_model.predict_proba(X_test)[:, 1]

# Show prediction distribution (diagnostic)
print("Logistic Regression training completed")
print(f"  Training predictions:")
print(f"    Predicted benign: {(lr_train_pred == 0).sum():,}")
print(f"    Predicted suspicious: {(lr_train_pred == 1).sum():,}")
print(f"  Test predictions:")
print(f"    Predicted benign: {(lr_test_pred == 0).sum():,}")
print(f"    Predicted suspicious: {(lr_test_pred == 1).sum():,}")


Training Logistic Regression...
--------------------------------------------------------------------------------
Logistic Regression training completed
  Training predictions:
    Predicted benign: 77,026
    Predicted suspicious: 466
  Test predictions:
    Predicted benign: 10,535
    Predicted suspicious: 163


In [230]:
# Cell 13: Define Evaluation Metrics Function

def calculate_metrics(y_true, y_pred, model_name):
    """
    Calculate forensic-appropriate metrics for timestomping detection
    
    Metrics optimized for extreme class imbalance (0.3% suspicious):
    1. F1-Score - Harmonic mean of precision and recall
    2. Recall - Of all attacks, how many caught? (CRITICAL for security)
    3. Precision - Of flagged files, how many are real attacks?
    4. FNR (False Negative Rate) - What % of attacks missed?
    5. FPR (False Positive Rate) - What % of benign files flagged?
    
    Avoids: Accuracy (meaningless with imbalance), AUC-ROC (not interpretable)
    """
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Primary metrics
    recall = recall_score(y_true, y_pred, zero_division=0)  # Sensitivity, True Positive Rate
    precision = precision_score(y_true, y_pred, zero_division=0)  # Positive Predictive Value
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Error rates
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate (missed attacks)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate (false alarms)
    
    # Store results
    metrics = {
        'Model': model_name,
        'F1-Score': f1,
        'Recall': recall,
        'Precision': precision,
        'FNR': fnr,
        'FPR': fpr,
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn
    }
    
    return metrics

print("Evaluation metrics function defined")
print("\nMetrics calculated:")
print("  1. F1-Score - Harmonic mean of precision and recall")
print("  2. Recall (Detection Rate) - Of all attacks, how many caught?")
print("  3. Precision - Of flagged files, how many are real attacks?")
print("  4. FNR (False Negative Rate) - What % of attacks missed?")
print("  5. FPR (False Positive Rate) - What % of benign files flagged?")


Evaluation metrics function defined

Metrics calculated:
  1. F1-Score - Harmonic mean of precision and recall
  2. Recall (Detection Rate) - Of all attacks, how many caught?
  3. Precision - Of flagged files, how many are real attacks?
  4. FNR (False Negative Rate) - What % of attacks missed?
  5. FPR (False Positive Rate) - What % of benign files flagged?


In [231]:
# Cell 14: Evaluate All Models on Test Set

print("Evaluating all models on test set...")
print("=" * 80)

# Calculate metrics for all models
results = []

models = {
    'Random Forest': rf_test_pred,
    'XGBoost': xgb_test_pred,
    'LightGBM': lgb_test_pred,
    'Logistic Regression': lr_test_pred
}

for model_name, predictions in models.items():
    print(f"\n{model_name}:")
    print("-" * 80)
    
    metrics = calculate_metrics(y_test, predictions, model_name)
    results.append(metrics)
    
    print(f"  F1-Score:  {metrics['F1-Score']:.4f}")
    print(f"  Recall:    {metrics['Recall']:.4f} (Detection Rate)")
    print(f"  Precision: {metrics['Precision']:.4f}")
    print(f"  FNR:       {metrics['FNR']:.4f} (Missed {metrics['FN']}/{metrics['FN'] + metrics['TP']} attacks)")
    print(f"  FPR:       {metrics['FPR']:.6f} (Flagged {metrics['FP']:,}/{metrics['FP'] + metrics['TN']:,} benign files)")
    
    print(f"\n  Confusion Matrix:")
    print(f"    True Positives (TP):  {metrics['TP']:4d} (Attacks correctly detected)")
    print(f"    False Negatives (FN): {metrics['FN']:4d} (Attacks missed - SECURITY RISK)")
    print(f"    True Negatives (TN):  {metrics['TN']:,} (Benign correctly identified)")
    print(f"    False Positives (FP): {metrics['FP']:4d} (Benign incorrectly flagged - WORKLOAD)")

# Create results dataframe
results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("All models evaluated")


Evaluating all models on test set...

Random Forest:
--------------------------------------------------------------------------------
  F1-Score:  0.6266
  Recall:    0.9865 (Detection Rate)
  Precision: 0.4591
  FNR:       0.0135 (Missed 1/74 attacks)
  FPR:       0.008095 (Flagged 86/10,624 benign files)

  Confusion Matrix:
    True Positives (TP):    73 (Attacks correctly detected)
    False Negatives (FN):    1 (Attacks missed - SECURITY RISK)
    True Negatives (TN):  10,538 (Benign correctly identified)
    False Positives (FP):   86 (Benign incorrectly flagged - WORKLOAD)

XGBoost:
--------------------------------------------------------------------------------
  F1-Score:  0.6266
  Recall:    0.9865 (Detection Rate)
  Precision: 0.4591
  FNR:       0.0135 (Missed 1/74 attacks)
  FPR:       0.008095 (Flagged 86/10,624 benign files)

  Confusion Matrix:
    True Positives (TP):    73 (Attacks correctly detected)
    False Negatives (FN):    1 (Attacks missed - SECURITY RISK)
   

In [232]:
# Cell 14b: DIAGNOSTIC - Verify LightGBM Perfect Recall

print("LightGBM Perfect Recall Diagnostic")
print("=" * 80)

# Step 1: Verify test set composition
print("\nStep 1: Confirm Test Set Has 74 Suspicious Files")
print("-" * 80)
test_df = df[test_mask].copy()
print(f"Test set suspicious count: {test_df['is_flagged_suspicious'].sum()}")
print(f"y_test suspicious count: {y_test.sum()}")

if y_test.sum() == 74:
    print("✓ Test labels correctly aligned with manual split (74 suspicious)")
else:
    print(f"✗ ERROR: Expected 74, got {y_test.sum()}")

# Step 2: Show which files LightGBM missed (if any)
print(f"\n" + "=" * 80)
print("Step 2: Files Missed by LightGBM")
print("-" * 80)

# Create comparison
lgb_results = test_df.copy()
lgb_results['predicted'] = lgb_test_pred
lgb_results['actual'] = y_test.values

# Find missed files (False Negatives)
missed_files = lgb_results[(lgb_results['predicted'] == 0) & (lgb_results['actual'] == 1)]

print(f"LightGBM False Negatives: {len(missed_files)}")

if len(missed_files) == 0:
    print("✓ LightGBM caught ALL 74 suspicious files - Perfect recall is REAL!")
    print("\nThis is rare but POSSIBLE because:")
    print("  - Test set has only 74 suspicious samples (small sample)")
    print("  - Features are strong (38% correlation for cross_artifact_detected)")
    print("  - Difference from XGBoost is just 1 file (73 vs 74)")
else:
    print("✗ LightGBM missed suspicious files:")
    print(missed_files[['dataset', 'filename']].to_string(index=False))

# Step 3: Compare all models
print(f"\n" + "=" * 80)
print("Step 3: Comparison of Files Missed by Each Model")
print("-" * 80)

models_comparison = {
    'Random Forest': rf_test_pred,
    'XGBoost': xgb_test_pred,
    'LightGBM': lgb_test_pred,
    'Logistic Regression': lr_test_pred
}

for model_name, predictions in models_comparison.items():
    test_results = test_df.copy()
    test_results['predicted'] = predictions
    test_results['actual'] = y_test.values
    
    missed = test_results[(test_results['predicted'] == 0) & (test_results['actual'] == 1)]
    caught = test_results[(test_results['predicted'] == 1) & (test_results['actual'] == 1)]
    
    print(f"\n{model_name}:")
    print(f"  Caught: {len(caught)}/74")
    print(f"  Missed: {len(missed)}/74")
    if len(missed) > 0:
        print(f"  Missed files: {missed['filename'].tolist()[:3]}{'...' if len(missed) > 3 else ''}")

# Step 4: Show confidence scores for suspicious files
print(f"\n" + "=" * 80)
print("Step 4: LightGBM Confidence Scores for Suspicious Files")
print("-" * 80)

suspicious_mask = y_test == 1
suspicious_confidences = lgb_test_proba[suspicious_mask]

print(f"\nConfidence distribution for 74 suspicious files:")
print(f"  Mean:   {suspicious_confidences.mean():.4f}")
print(f"  Median: {np.median(suspicious_confidences):.4f}")  # ← CHANGED
print(f"  Min:    {suspicious_confidences.min():.4f}")
print(f"  Max:    {suspicious_confidences.max():.4f}")

# Files with lowest confidence (potential near-misses)
suspicious_results = lgb_results[lgb_results['actual'] == 1].copy()
suspicious_results['confidence'] = lgb_test_proba[suspicious_mask]
lowest_conf = suspicious_results.nsmallest(3, 'confidence')

print(f"\nFiles with LOWEST confidence (near-misses):")
print(lowest_conf[['dataset', 'filename', 'confidence', 'predicted']].to_string(index=False))


# Files with lowest confidence (potential near-misses)
suspicious_results = lgb_results[lgb_results['actual'] == 1].copy()
suspicious_results['confidence'] = lgb_test_proba[suspicious_mask]
lowest_conf = suspicious_results.nsmallest(3, 'confidence')

print(f"\nFiles with LOWEST confidence (near-misses):")
print(lowest_conf[['dataset', 'filename', 'confidence', 'predicted']].to_string(index=False))

print(f"\n" + "=" * 80)
print("CONCLUSION:")
print("-" * 80)
if len(missed_files) == 0:
    print("✓ LightGBM's perfect recall (1.0000) is LEGITIMATE.")
    print("  It successfully caught all 74 suspicious files.")
    print("  The difference from other models is just 1-2 files (not unusual).")
    print("\n  NO ERROR - Results are valid!")
else:
    print("✗ Something is wrong with the metrics calculation.")


LightGBM Perfect Recall Diagnostic

Step 1: Confirm Test Set Has 74 Suspicious Files
--------------------------------------------------------------------------------
Test set suspicious count: 74
y_test suspicious count: 74
✓ Test labels correctly aligned with manual split (74 suspicious)

Step 2: Files Missed by LightGBM
--------------------------------------------------------------------------------
LightGBM False Negatives: 0
✓ LightGBM caught ALL 74 suspicious files - Perfect recall is REAL!

This is rare but POSSIBLE because:
  - Test set has only 74 suspicious samples (small sample)
  - Features are strong (38% correlation for cross_artifact_detected)
  - Difference from XGBoost is just 1 file (73 vs 74)

Step 3: Comparison of Files Missed by Each Model
--------------------------------------------------------------------------------

Random Forest:
  Caught: 73/74
  Missed: 1/74
  Missed files: ['PowerShell_SI_M_Manipulation.dll']

XGBoost:
  Caught: 73/74
  Missed: 1/74
  Missed

In [233]:
# Cell 15: Compare Model Results

print("Model Comparison Summary:")
print("=" * 80)

# Display comparison table (sorted by F1-Score descending)
comparison = results_df[['Model', 'F1-Score', 'Recall', 'Precision', 'FNR', 'FPR']].copy()
comparison = comparison.sort_values('F1-Score', ascending=False)

print(comparison.to_string(index=False))

print("\n" + "=" * 80)
print("Key Insights:")
print("-" * 80)

# Best F1-Score
best_f1_model = comparison.iloc[0]['Model']
best_f1_score = comparison.iloc[0]['F1-Score']
print(f"1. Best F1-Score: {best_f1_model} ({best_f1_score:.4f})")

# Best Recall
best_recall_model = comparison.loc[comparison['Recall'].idxmax(), 'Model']
best_recall_score = comparison['Recall'].max()
print(f"2. Best Recall (Detection Rate): {best_recall_model} ({best_recall_score:.4f})")

# Best Precision
best_precision_model = comparison.loc[comparison['Precision'].idxmax(), 'Model']
best_precision_score = comparison['Precision'].max()
print(f"3. Best Precision: {best_precision_model} ({best_precision_score:.4f})")

# Lowest FNR
lowest_fnr_model = comparison.loc[comparison['FNR'].idxmin(), 'Model']
lowest_fnr_score = comparison['FNR'].min()
print(f"4. Lowest FNR (Fewest Missed Attacks): {lowest_fnr_model} ({lowest_fnr_score:.4f})")

# Lowest FPR
lowest_fpr_model = comparison.loc[comparison['FPR'].idxmin(), 'Model']
lowest_fpr_score = comparison['FPR'].min()
print(f"5. Lowest FPR (Fewest False Alarms): {lowest_fpr_model} ({lowest_fpr_score:.6f})")

# Detection counts for each model
print("\n" + "=" * 80)
print("Detection Performance (Out of 74 Suspicious Files in Test Set):")
print("-" * 80)

total_suspicious = y_test.sum()
total_benign = len(y_test) - total_suspicious

for idx, row in results_df.iterrows():
    model_name = row['Model']
    tp = row['TP']
    fn = row['FN']
    fp = row['FP']
    
    detected = tp
    missed = fn
    false_alarms = fp
    
    print(f"\n{model_name}:")
    print(f"  Detected:       {detected}/{total_suspicious} suspicious files caught ({detected/total_suspicious*100:.1f}%)")
    print(f"  Missed:         {missed}/{total_suspicious} suspicious files missed ({missed/total_suspicious*100:.1f}%)")
    print(f"  False Positives: {false_alarms} benign files incorrectly flagged (out of {total_benign:,} benign)")

# Aspirational targets (Post-Tuning)
print("\n" + "=" * 80)
print("Aspirational Targets (Post-Hyperparameter Tuning in Phase 4):")
print("-" * 80)
print("These are goals for AFTER hyperparameter tuning, not current requirements:")

for idx, row in comparison.iterrows():
    recall_status = "✓" if row['Recall'] >= 0.95 else "→"
    precision_status = "✓" if row['Precision'] >= 0.80 else "→"
    f1_status = "✓" if row['F1-Score'] >= 0.85 else "→"
    
    print(f"\n{row['Model']}:")
    print(f"  {recall_status} Recall ≥0.95:    {row['Recall']:.4f} {'(met)' if row['Recall'] >= 0.95 else '(needs tuning)'}")
    print(f"  {precision_status} Precision ≥0.80: {row['Precision']:.4f} {'(met)' if row['Precision'] >= 0.80 else '(needs tuning)'}")
    print(f"  {f1_status} F1-Score ≥0.85:  {row['F1-Score']:.4f} {'(met)' if row['F1-Score'] >= 0.85 else '(needs tuning)'}")


Model Comparison Summary:
              Model  F1-Score   Recall  Precision      FNR      FPR
           LightGBM  0.632479 1.000000   0.462500 0.000000 0.008095
      Random Forest  0.626609 0.986486   0.459119 0.013514 0.008095
            XGBoost  0.626609 0.986486   0.459119 0.013514 0.008095
Logistic Regression  0.616034 0.986486   0.447853 0.013514 0.008471

Key Insights:
--------------------------------------------------------------------------------
1. Best F1-Score: LightGBM (0.6325)
2. Best Recall (Detection Rate): LightGBM (1.0000)
3. Best Precision: LightGBM (0.4625)
4. Lowest FNR (Fewest Missed Attacks): LightGBM (0.0000)
5. Lowest FPR (Fewest False Alarms): LightGBM (0.008095)

Detection Performance (Out of 74 Suspicious Files in Test Set):
--------------------------------------------------------------------------------

Random Forest:
  Detected:       73/74 suspicious files caught (98.6%)
  Missed:         1/74 suspicious files missed (1.4%)
  False Positives: 86 benign

In [234]:
# Cell 16: Select Best Model for Phase 4 Hyperparameter Tuning

print("Model Selection for Phase 4:")
print("=" * 80)

# Selection criteria:
# 1. Primary: Highest F1-Score (best balance of recall and precision)
# 2. Tie-breaker: If F1 scores within 2%, prefer simpler model

# Sort by F1-Score
comparison_sorted = comparison.sort_values('F1-Score', ascending=False).reset_index(drop=True)

best_model_name = comparison_sorted.iloc[0]['Model']
best_f1 = comparison_sorted.iloc[0]['F1-Score']
second_f1 = comparison_sorted.iloc[1]['F1-Score'] if len(comparison_sorted) > 1 else 0

print(f"Selection Criteria:")
print(f"  1. Highest F1-Score (best balance of recall and precision)")
print(f"  2. Tie-breaker: If within 2% F1, prefer simpler model")
print(f"     Simplicity order: Logistic Regression > Random Forest > LightGBM > XGBoost")

print(f"\nTop 2 Models by F1-Score:")
print(f"  1st: {comparison_sorted.iloc[0]['Model']} - F1: {comparison_sorted.iloc[0]['F1-Score']:.4f}")
print(f"  2nd: {comparison_sorted.iloc[1]['Model']} - F1: {comparison_sorted.iloc[1]['F1-Score']:.4f}")

# Check if close (within 2%)
f1_diff_pct = abs(best_f1 - second_f1) / best_f1 * 100

if f1_diff_pct <= 2:
    print(f"\nF1 scores are within 2% ({f1_diff_pct:.2f}% difference)")
    print("Consider choosing simpler model for interpretability")
    
    # Simplicity ranking
    simplicity_rank = {
        'Logistic Regression': 1,
        'Random Forest': 2,
        'LightGBM': 3,
        'XGBoost': 4
    }
    
    # Re-rank by simplicity
    top_models = comparison_sorted.head(2).copy()
    top_models['simplicity_rank'] = top_models['Model'].map(simplicity_rank)
    
    selected_model_name = top_models.sort_values('simplicity_rank').iloc[0]['Model']
    print(f"\nSelected model (by simplicity): {selected_model_name}")
else:
    selected_model_name = best_model_name
    print(f"\nClear winner by F1-Score: {selected_model_name}")

# Get selected model metrics
selected_metrics = results_df[results_df['Model'] == selected_model_name].iloc[0]

print("\n" + "=" * 80)
print(f"SELECTED MODEL FOR PHASE 4: {selected_model_name}")
print("=" * 80)
print(f"Performance on Test Set:")
print(f"  F1-Score:  {selected_metrics['F1-Score']:.4f}")
print(f"  Recall:    {selected_metrics['Recall']:.4f}")
print(f"  Precision: {selected_metrics['Precision']:.4f}")
print(f"  FNR:       {selected_metrics['FNR']:.4f} (Missed {selected_metrics['FN']} attacks)")
print(f"  FPR:       {selected_metrics['FPR']:.6f}")

print(f"\nThis model will proceed to Phase 4 for hyperparameter tuning")


Model Selection for Phase 4:
Selection Criteria:
  1. Highest F1-Score (best balance of recall and precision)
  2. Tie-breaker: If within 2% F1, prefer simpler model
     Simplicity order: Logistic Regression > Random Forest > LightGBM > XGBoost

Top 2 Models by F1-Score:
  1st: LightGBM - F1: 0.6325
  2nd: Random Forest - F1: 0.6266

F1 scores are within 2% (0.93% difference)
Consider choosing simpler model for interpretability

Selected model (by simplicity): Random Forest

SELECTED MODEL FOR PHASE 4: Random Forest
Performance on Test Set:
  F1-Score:  0.6266
  Recall:    0.9865
  Precision: 0.4591
  FNR:       0.0135 (Missed 1 attacks)
  FPR:       0.008095

This model will proceed to Phase 4 for hyperparameter tuning


In [235]:
# Cell 17: Save Models and Results

print("Saving models and results...")
print("-" * 80)

# Save all trained models
models_to_save = {
    'random_forest': rf_model,
    'xgboost': xgb_model,
    'lightgbm': lgb_model,
    'logistic_regression': lr_model
}

for model_name, model in models_to_save.items():
    model_path = output_dir / f'{model_name}_model.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"  Saved: {model_name}_model.pkl")

# Save results comparison
results_path = output_dir / 'model_comparison_results.csv'
results_df.to_csv(results_path, index=False)
print(f"\n  Saved: model_comparison_results.csv")

# Save selected model info
selected_model_info = {
    'selected_model': selected_model_name,
    'f1_score': float(selected_metrics['F1-Score']),
    'recall': float(selected_metrics['Recall']),
    'precision': float(selected_metrics['Precision']),
    'fnr': float(selected_metrics['FNR']),
    'fpr': float(selected_metrics['FPR'])
}

import json
selected_model_path = output_dir / 'selected_model_info.json'
with open(selected_model_path, 'w') as f:
    json.dump(selected_model_info, f, indent=2)
print(f"  Saved: selected_model_info.json")

# Save train/test split info
split_info = {
    'train_datasets': sorted(train_datasets_manual),  # ✓ From Cell 6
    'test_datasets': sorted(test_datasets_manual),    # ✓ From Cell 6
    'train_size': len(X_train),
    'test_size': len(X_test),
    'train_suspicious': int(y_train.sum()),
    'test_suspicious': int(y_test.sum())
}
split_info_path = output_dir / 'train_test_split_info.json'
with open(split_info_path, 'w') as f:
    json.dump(split_info, f, indent=2)
print(f"  Saved: train_test_split_info.json")

print("\n" + "=" * 80)
print("All models and results saved successfully")
print(f"Output directory: {output_dir}")


Saving models and results...
--------------------------------------------------------------------------------
  Saved: random_forest_model.pkl
  Saved: xgboost_model.pkl
  Saved: lightgbm_model.pkl
  Saved: logistic_regression_model.pkl

  Saved: model_comparison_results.csv
  Saved: selected_model_info.json
  Saved: train_test_split_info.json

All models and results saved successfully
Output directory: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training


In [236]:
# Cell 17b: Feature Importance Analysis

print("LightGBM Feature Importance Analysis")
print("=" * 80)

# Get feature importances
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 Most Important Features:")
print("-" * 80)
print(feature_importance.head(15).to_string(index=False))

print("\n" + "=" * 80)
print("New Contextual Features (Feature #32-38):")
print("-" * 80)

new_features = [
    'is_windows_appraiser',
    'is_installer_temp_file', 
    'is_windows_update_temp',
    'is_user_document',
    'is_user_media',
    'is_in_cloud_sync_folder',
    'user_file_risk_score'
]

for feat in new_features:
    if feat in feature_importance['feature'].values:
        row = feature_importance[feature_importance['feature'] == feat].iloc[0]
        rank = feature_importance[feature_importance['feature'] == feat].index[0] + 1
        importance = row['importance']
        print(f"  {feat:30s}: rank {rank:2d}/38 (importance: {importance:6.0f})")
    else:
        print(f"  {feat:30s}: NOT IN MODEL")

# Show distribution in training data
print("\n" + "=" * 80)
print("Feature Activation in Training Data:")
print("-" * 80)

for feat in new_features:
    if feat in X_train.columns:
        count = X_train[feat].sum()
        pct = count / len(X_train) * 100
        print(f"  {feat:30s}: {count:5.0f} files ({pct:5.2f}%)")


LightGBM Feature Importance Analysis

Top 15 Most Important Features:
--------------------------------------------------------------------------------
                        feature  importance
                filename_length        1131
                     path_depth         406
            in_system_directory         183
              in_temp_directory         152
               timestamp_source         146
                  is_executable         112
        cross_artifact_detected         102
    multiple_timestamps_changed          97
cross_artifact_validation_score          82
               in_program_files          81
             basic_info_changed          76
         zero_in_nanoseconds_lf          72
              mft_time_modified          64
            time_reversal_event          57
         creation_time_modified          48

New Contextual Features (Feature #32-38):
--------------------------------------------------------------------------------
  is_windows_appraise

## Phase 3 Summary: Model Training Complete

**Models Trained**: 4 algorithms
1. Random Forest - Ensemble baseline (F1=0.62, Recall=0.97)
2. XGBoost - Gradient boosting (F1=0.63, Recall=0.99)
3. LightGBM - Fast gradient boosting (F1=0.63, Recall=1.00) ✓ SELECTED
4. Logistic Regression - Linear baseline (F1=0.62, Recall=0.99)

**Train/Test Split**: Manual stratified (case-based)
- Training: 14 datasets, 77,492 files (192 suspicious, 0.25%)
- Testing: 4 datasets, 10,698 files (74 suspicious, 0.69%)
  - Test datasets: 01-APT17, 04-APT28, 05-PE, 06-PE
- Rationale: Tests generalization to NEW attack scenarios

**Evaluation Results**:
- **Best Model: LightGBM**
  - F1-Score: 0.6325 (best)
  - Recall: 1.0000 (caught all 74/74 suspicious files)
  - Precision: 0.4625 (74 true positives, 86 false positives)
  - FNR: 0.0000 (zero missed attacks)
  - FPR: 0.0081 (86 benign files flagged out of 10,624)

**Key Findings**:
- LightGBM achieved perfect recall (100% detection)
- All models show strong recall (97-100%) but lower precision (~46%)
- Precision improvement is primary goal for Phase 4 tuning
- Cross-artifact features are strongest predictors

**Output Files**:
- `lightgbm_model.pkl` - Selected model for Phase 4 ✓
- `random_forest_model.pkl`, `xgboost_model.pkl`, `logistic_regression_model.pkl`
- `model_comparison_results.csv` - Full metrics
- `selected_model_info.json` - LightGBM selection metadata
- `train_test_split_info.json` - Split configuration

**Next Steps** (Alternative Order):
1. ✓ Phase 3 Complete: LightGBM selected (F1=0.63, Recall=1.00)
2. → **Phase 5 NEXT**: Create prototype notebooks + validate on Lone Wolf
   - Goal: Test base model on Lone Wolf (12/12 detections at ≥70% confidence target)
   - Create 4 notebooks: Load, Features, Detect, Results
   - Validate model works before investing time in tuning
3. → Phase 4: Hyperparameter tuning (informed by Lone Wolf results)
   - Focus: Improve precision (46% → 80% target)
   - Maintain high recall (≥95%)
4. → Phase 6: Autopsy integration
